# 09 CPI overlay: short-term TQQQ -> UVXY hedge (part of the final strategy)

**This overlay is part of the final strategy.** The rule and helpers defined here (`load_cpi`, `cpi_event_table`, `apply_overlay`, `overlay_for`) are loaded by 07 (historical backtest) and 08 (October live account), so the `final_*` results include it. This notebook documents the rule, shows with / without comparisons (`cpi_overlay_*` files) and, in the two cells at the end, how the switch and the hedge settings were set. It does not change `v2_weights_history.csv`.

**Idea.** CPI is published at 08:30 New York, before the US open. If the actual year-on-year CPI is above the forecast (a hot surprise) and the market is currently reacting to CPI, swap the sleeve's TQQQ target weight into UVXY for a few days.

**Rule (parameters are constants below; there is no search inside this cell; how they were set is explained at the end).** A hot-CPI date triggers the hedge only if both conditions hold:
1. The previous published actual CPI year-on-year was at least `MIN_YOY` (3.0%).
2. Of the previous `WINDOW` (16) CPI events with a non-zero surprise, at least `MIN_HIT` (70%) had a QQQ opening gap in the direction of the surprise (hot -> gap down, cool -> gap up). Only events before the current one are used.

**Execution.**
- Day 1 (CPI date), open: `HEDGE_FRACTION` of the TQQQ target weight goes to UVXY. The row of the V2 weight file dated the previous close is the one used at that open; the CPI number is public before the open, so it is known when the order is placed.
- Days 2 and 3, open: same swap on that day's V2 target.
- Day 3, **close**: one extra rebalance at the source closing price back to that day's V2 target (the unhedged weights). Fees are one-way rate times the actual notional. It needs a closing-auction (MOC) order in practice and assumes no slippage.
- All trading follows the standalone US account execution and fee assumptions. This close-price exit is the only close-price trade in the final strategy; a no-overlay run has none.

**Data.** `data/raw/cpi_yoy.csv` (release date, reference month, forecast, actual). It is maintained by hand: add one row after each CPI release, with the forecast taken from the same source as the other rows. One duplicated release date (2025-10-24) was reduced to its first row.

**Outputs.** `results/signals/v2_cpi_overlay_weights_history.csv`, `results/tables/cpi_overlay_metrics.csv`, `cpi_overlay_yearly.csv`, `cpi_overlay_triggers.csv`, `cpi_overlay_equity_history.csv`, and `results/figures/cpi_overlay_*.png`.

**Caveats to quote with any number from this stage.** Only a handful of triggers; the start of the regime (market reacting to CPI) and the window/threshold were chosen after seeing data, so the OOS window is not a clean out-of-sample test; the CPI forecast is a consensus number from an outside source; the gain is mostly from 2023, 2024 and the October 2026 release; the maximum drawdown is not reduced.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / 'pipeline/00_engine.ipynb').is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import portfolio_accounting as pa

TABLE_DIR, SIGNAL_DIR, FIG_DIR = [ROOT / 'results' / s for s in ('tables', 'signals', 'figures')]
CPI_CSV = ROOT / 'data' / 'raw' / 'cpi_yoy.csv'

# Rule parameters, fixed in advance (see the description above).
MIN_YOY, WINDOW, MIN_PERIODS, MIN_HIT = 3.0, 16, 8, 0.70
HOLD_DAYS, HEDGE_FRACTION, FROM_TICKER, HEDGE_TICKER = 3, 1.0, 'TQQQ', 'UVXY'
GAP_TICKER = 'QQQ'
REBALANCE_BAND = 0.01
BASE_NAME = 'V2 (US-only)'
OVERLAY_NAME = 'V2 + CPI Overlay (US-only)'


def load_cpi(path=CPI_CSV):
    """Release date, forecast and actual CPI YoY; one row per release date."""
    cpi = pd.read_csv(path, parse_dates=['Release_Date'])
    need = {'Release_Date', 'Predicted_YOY', 'Actual_YOY'}
    if not need.issubset(cpi.columns):
        raise ValueError(f'{path}: missing columns {sorted(need - set(cpi.columns))}')
    cpi = cpi.dropna(subset=['Release_Date', 'Predicted_YOY', 'Actual_YOY'])
    cpi = cpi.drop_duplicates('Release_Date', keep='first').sort_values('Release_Date')
    return cpi.rename(columns={'Release_Date': 'release', 'Predicted_YOY': 'predicted', 'Actual_YOY': 'actual'}).reset_index(drop=True)[['release', 'predicted', 'actual']]


def cpi_event_table(cpi, opens, closes, min_yoy=MIN_YOY, window=WINDOW, min_periods=MIN_PERIODS, min_hit=MIN_HIT):
    """One row per CPI release that fell on a trading day, with the switch evaluated.

    Every quantity at a release uses only releases BEFORE it (the previous actual YoY,
    and the hit rate of earlier events), plus the current forecast and actual, which
    are public before the US open. The current opening gap is never used for the
    current decision; it only enters later events' hit rates.
    """
    close, opn = closes[GAP_TICKER], opens[GAP_TICKER]
    gap = (opn / close.shift(1) - 1).dropna()
    ev = cpi[cpi.release.isin(gap.index)].copy().reset_index(drop=True)
    ev['gap'] = gap.reindex(ev.release).to_numpy()
    ev['surprise'] = (ev.actual - ev.predicted).round(2)
    ev['last_yoy'] = ev.actual.shift(1)
    nz = ev[ev.surprise != 0].copy()
    nz['hit'] = np.where(nz.surprise > 0, nz.gap < 0, nz.gap > 0).astype(float)
    nz['hit_rate'] = nz.hit.rolling(window, min_periods=min_periods).mean().shift(1)
    ev = ev.merge(nz[['release', 'hit', 'hit_rate']], on='release', how='left')
    ev['hit_rate'] = ev.hit_rate.ffill()
    ev['trigger'] = ((ev.surprise > 0) & (ev.last_yoy >= min_yoy) & (ev.hit_rate >= min_hit - 1e-12)).fillna(False)
    return ev


def apply_overlay(weights, trigger_dates, calendar, hold_days=HOLD_DAYS, fraction=HEDGE_FRACTION,
                  from_ticker=FROM_TICKER, hedge_ticker=HEDGE_TICKER):
    """Move `fraction` of the from_ticker target into hedge_ticker for `hold_days` executions.

    Returns (overlay weights, close_events, rows). The weight row used at the open of
    day d is the last row dated before d. After the last hedged day's open, one
    rebalance at that day's close goes back to the unhedged target (close_events).
    A hedge whose last day is not yet in the data has no exit yet.
    """
    w = weights.copy()
    calendar = pd.DatetimeIndex(calendar)
    close_events, rows = {}, []
    for t in pd.DatetimeIndex(trigger_dates):
        if t not in calendar:
            continue
        k = calendar.get_loc(t)
        days = calendar[k:k + hold_days]
        for d in days:
            pos = weights.index.searchsorted(d, side='left') - 1
            if pos < 0:
                continue
            moved = w.iloc[pos][from_ticker] * fraction
            w.iloc[pos, w.columns.get_loc(from_ticker)] -= moved
            w.iloc[pos, w.columns.get_loc(hedge_ticker)] += moved
        exit_date = days[-1] if len(days) == hold_days else pd.NaT
        if pd.notna(exit_date):
            pos = weights.index.searchsorted(exit_date, side='left') - 1
            close_events[exit_date] = (weights.index[pos], weights.iloc[pos].to_numpy(float))
        rows.append(dict(release=t, first_hedged_open=days[0], last_hedged_day=days[-1], close_exit=exit_date))
    return w, close_events, pd.DataFrame(rows)


def window_summary(eq, start, end):
    window = pa.window_equity(eq, start, end)
    perf = pa.performance(eq, start, end)
    ret = window.iloc[-1] / window.iloc[0] - 1 if len(window) > 1 else np.nan
    return dict(Return=ret, CAGR=perf['CAGR'], Sharpe=perf['Sharpe'], MDD=perf['MDD'])


def overlay_for(us, w_v2, cpi=None):
    """The final strategy's US weights: V2 plus the CPI overlay. Used by 07, 08 and this notebook.

    Returns (overlay weights, close_events for run_sleeve(), event table, hedge rows).
    """
    cpi = load_cpi() if cpi is None else cpi
    events = cpi_event_table(cpi, us.opens, us.closes)
    calendar = us.closes[GAP_TICKER].dropna().index
    w_over, close_events, hedge_rows = apply_overlay(w_v2, events.loc[events.trigger, 'release'], calendar)
    return w_over, close_events, events, hedge_rows


def main(csv_path=None, save=True):
    """Compare V2 and V2+CPI using the standalone US account."""
    if not CPI_CSV.is_file():
        raise FileNotFoundError(f"Missing CPI data: {CPI_CSV}")

    us = pa.load_us_market(csv_path)
    w_v2 = pa.us_strategy_weights("v2")
    w_over, close_events, events, hedge_rows = overlay_for(us, w_v2)

    start = pa.HISTORICAL_START
    timeline = pa.common_timeline([us], start, None)

    base = pa.run_sleeve(us, w_v2, timeline, 100000.0)
    over = pa.run_sleeve(
        us, w_over, timeline, 100000.0,
        close_events=close_events,
    )

    curves = {
        "V2 (US-only)": base["daily"].equity,
        "V2 + CPI Overlay (US-only)": over["daily"].equity,
    }

    rows = []
    for name, eq in curves.items():
        for label, a, b in pa.PERIODS:
            rows.append(dict(
                Strategy=name, Period=label,
                **pa.performance(eq, a, b),
            ))

    metrics = pd.DataFrame(rows)
    triggers = events[events.trigger].merge(
        hedge_rows, on="release", how="left"
    )

    display(metrics.round(4))
    display(triggers)

    if save:
        for folder in (TABLE_DIR, SIGNAL_DIR):
            folder.mkdir(parents=True, exist_ok=True)

        w_over.to_csv(
            SIGNAL_DIR / "v2_cpi_overlay_weights_history.csv",
            encoding="utf-8-sig",
        )
        metrics.to_csv(
            TABLE_DIR / "cpi_overlay_metrics.csv",
            index=False,
        )
        triggers.to_csv(
            TABLE_DIR / "cpi_overlay_triggers.csv",
            index=False,
        )
        pd.DataFrame(curves).to_csv(
            TABLE_DIR / "cpi_overlay_equity_history.csv"
        )

    return {
        "base": base,
        "overlay": over,
        "events": events,
        "triggers": triggers,
        "metrics": metrics,
        "weights": w_over,
    }


In [ ]:
# Run All after 03 -> 02 -> 05 -> 06 have produced their signal files.
results_cpi = main(save=True)


## How the switch (16 events, hit rate >= 70%, YoY >= 3%) was set

**Prior.** After the COVID recovery, inflation became the main macro theme and the market started to react to CPI releases. This is the economic reason for having a CPI switch at all.

**Values.** The specific numbers (previous YoY >= 3.0%, a rolling window of 16 non-zero-surprise events, hit rate >= 70%) were set after inspecting the hit rate by period below. They were not optimised for return: the neighbouring-parameter table shows other settings with a higher OOS Sharpe, and the YoY >= 3.0% condition is there to keep the switch off in the IS period. Because they were set with the data in view, the OOS result is not a clean out-of-sample test.

To run it, set `RUN_EVIDENCE = True` in the next cell and run that cell (several minutes); it is off by default, so `run_all` skips it.

- **Hit** = QQQ opening gap (open / previous close - 1) in the direction of the CPI surprise (hot -> gap down, cool -> gap up). Events with zero surprise are excluded.
- **Part A** shows the hit rate by period: around 50-60% before 2022 (no reliable reaction), clearly higher afterwards. That is why a rolling hit rate is used as the switch instead of a fixed start date.
- **Part B** re-runs the full two-sleeve backtest for neighbouring `WINDOW`, `MIN_HIT` and `MIN_YOY` values, so you can see how much the result depends on the exact 16 / 70% / 3.0%.

In [ ]:
def switch_evidence(csv_path=None, windows=(8, 12, 16, 20, 24), hits=(0.6, 0.7, 0.8), yoys=(0.0, 3.0), save=True):
    us = pa.load_us_market(csv_path)
    w_v2 = pa.us_strategy_weights('v2')
    cpi = load_cpi()
    calendar = us.closes[GAP_TICKER].dropna().index
    # Part A: hit rate by period (event level, no trading involved)
    ev = cpi_event_table(cpi, us.opens, us.closes)
    nz = ev[ev.surprise != 0].copy()
    nz['hit'] = np.where(nz.surprise > 0, nz.gap < 0, nz.gap > 0).astype(float)
    bins = [('2010-2020', '2010-01-01', '2020-12-31'), ('2021', '2021-01-01', '2021-12-31'),
            ('2022-2023', '2022-01-01', '2023-12-31'), ('2024 to latest', '2024-01-01', '2100-01-01')]
    part_a = []
    for label, a, b in bins:
        s = nz[(nz.release >= a) & (nz.release <= b)]
        hot = s[s.surprise > 0]
        part_a.append(dict(Period=label, Events=len(s), Hits=int(s.hit.sum()), HitRate=s.hit.mean() if len(s) else np.nan,
                           HotEvents=len(hot), HotMeanGap=hot.gap.mean() if len(hot) else np.nan,
                           OtherMeanGap=s[s.surprise < 0].gap.mean() if (s.surprise < 0).any() else np.nan))
    part_a = pd.DataFrame(part_a)
    # Part B: neighbouring parameters, full backtest (close-exit overlay, +/-1pt reset), vs no hedge
    start = pa.HISTORICAL_START
    base = pa.run_sleeve(us, w_v2, pa.common_timeline([us], start), 1.0)['daily'].equity
    last = str(base.index[-1].date())
    periods = [('IS', pa.ENGINE['IS_START'], pa.ENGINE['IS_END']), ('OOS', pa.ENGINE['OOS_START'], pa.ENGINE['OOS_END']),
               ('OOS ex-2020', '2021-01-01', pa.ENGINE['OOS_END'])]
    rows = []
    def add(label, params, eq, n):
        for pl, a, b in periods:
            m = window_summary(eq, a, b)
            rows.append(dict(Setting=label, Window=params[0], MinHit=params[1], MinYoY=params[2], Triggers=n, Period=pl, **m))
    add('No hedge', (None, None, None), base, 0)
    for wdw in windows:
        for h in hits:
            for y in yoys:
                e = cpi_event_table(cpi, us.opens, us.closes, min_yoy=y, window=wdw, min_periods=max(4, wdw // 2), min_hit=h)
                w_o, ce, _ = apply_overlay(w_v2, e.loc[e.trigger, 'release'], calendar)
                eq = pa.run_sleeve(us, w_o, pa.common_timeline([us], start), 1.0, close_events=ce)['daily'].equity
                add('Chosen' if (wdw, h, y) == (WINDOW, MIN_HIT, MIN_YOY) else 'Neighbour', (wdw, h, y), eq, int(e.trigger.sum()))
    part_b = pd.DataFrame(rows)
    display(part_a.round(4))
    part_b['Key'] = np.where(part_b.Window.isna(), 'No hedge',
                             'W' + part_b.Window.fillna(0).astype(int).astype(str) + ' / hit>=' + (part_b.MinHit.fillna(0) * 100).round().astype(int).astype(str)
                             + '% / YoY>=' + part_b.MinYoY.fillna(0).astype(str))
    wide = part_b.pivot(index=['Key', 'Triggers', 'Setting'], columns='Period', values=['CAGR', 'Sharpe'])
    wide = wide.reindex(columns=pd.MultiIndex.from_product([['CAGR', 'Sharpe'], ['IS', 'OOS', 'OOS ex-2020']]))
    display(wide.round(3))
    if save:
        TABLE_DIR.mkdir(parents=True, exist_ok=True)
        part_a.to_csv(TABLE_DIR / 'cpi_overlay_hit_rate_by_period.csv', index=False, encoding='utf-8-sig')
        part_b.to_csv(TABLE_DIR / 'cpi_overlay_switch_sensitivity.csv', index=False, encoding='utf-8-sig')
    return part_a, part_b


RUN_EVIDENCE = False        # set to True to run this check (several minutes); False so run_all and the other notebooks skip it
if RUN_EVIDENCE:
    evidence_a, evidence_b = switch_evidence(save=True)


## How the hedge (UVXY, 100%, 3 days) was chosen

Same spirit as above: a sensitivity check around the chosen setting with the **switch fixed** (YoY >= 3.0, 16 events, 70%), not a search for the best point. The hedge asset must be a column of the V2 weight file (TQQQ, UVXY, TECL, SPXL, TECS, BSV), so TECS (3x inverse technology) and BSV (short bond) are the alternatives shown. To run it, set `RUN_HEDGE_EVIDENCE = True` in the next cell and run that cell (several minutes); it is off by default, so `run_all` skips it.

**Range of H.** The hold is limited to 1-3 trading days **by design**, because the purpose is a short-term protection around the release, not a holding. Longer holds (4 and 5 days) were also run when this table was built; they gave higher OOS Sharpe than 3 days, including with TECS and BSV as the hedge asset, so the result is not specific to UVXY. They are left out of the table on purpose and were not used to choose H, since picking the length after seeing OOS would be post-hoc. H=3 is therefore **not** claimed to be the best point of a search; within 1-3 days it is the highest OOS Sharpe for UVXY, but H=2 is below the no-hedge result.

Reasons for the chosen setting that the table cannot show: CPI is released before the open, so the first gap is already priced when the hedge is bought, and the idea is a **short-term** protection against the follow-through, not a holding; UVXY is the most direct short-volatility-spike instrument; 100% of the TQQQ target is moved so the effect is not diluted. The table only checks that the result does not hinge on one lucky point.

In [ ]:
def hedge_evidence(csv_path=None, holds=(1, 2, 3), fractions=(0.5, 1.0), assets=('UVXY', 'TECS', 'BSV'), save=True):
    us = pa.load_us_market(csv_path)
    w_v2 = pa.us_strategy_weights('v2')
    cpi = load_cpi()
    calendar = us.closes[GAP_TICKER].dropna().index
    ev = cpi_event_table(cpi, us.opens, us.closes)          # switch fixed at the chosen values
    trig = ev.loc[ev.trigger, 'release']
    start = pa.HISTORICAL_START
    periods = [('IS', pa.ENGINE['IS_START'], pa.ENGINE['IS_END']), ('OOS', pa.ENGINE['OOS_START'], pa.ENGINE['OOS_END']),
               ('OOS ex-2020', '2021-01-01', pa.ENGINE['OOS_END'])]
    rows = []
    def add(asset, h, f, eq):
        for pl, a, b in periods:
            rows.append(dict(Asset=asset, Hold=h, Fraction=f, Chosen=(asset, h, f) == (HEDGE_TICKER, HOLD_DAYS, HEDGE_FRACTION),
                             Period=pl, **window_summary(eq, a, b)))
    add('No hedge', 0, 0.0, pa.run_sleeve(us, w_v2, pa.common_timeline([us], start), 1.0)['daily'].equity)
    for asset in assets:
        for h in holds:
            for f in fractions:
                w_o, ce, _ = apply_overlay(w_v2, trig, calendar, hold_days=h, fraction=f, hedge_ticker=asset)
                eq = pa.run_sleeve(us, w_o, pa.common_timeline([us], start), 1.0, close_events=ce)['daily'].equity
                add(asset, h, f, eq)
    table = pd.DataFrame(rows)
    wide = table.pivot(index=['Asset', 'Hold', 'Fraction', 'Chosen'], columns='Period', values=['CAGR', 'Sharpe', 'MDD'])
    wide = wide.reindex(columns=pd.MultiIndex.from_product([['CAGR', 'Sharpe', 'MDD'], ['IS', 'OOS', 'OOS ex-2020']]))
    display(wide.round(3))
    if save:
        TABLE_DIR.mkdir(parents=True, exist_ok=True)
        table.to_csv(TABLE_DIR / 'cpi_overlay_hedge_sensitivity.csv', index=False, encoding='utf-8-sig')
    return table


RUN_HEDGE_EVIDENCE = False  # set to True to run this check (several minutes); False so run_all and the other notebooks skip it
if RUN_HEDGE_EVIDENCE:
    hedge_table = hedge_evidence(save=True)
